"""
00_data_cleaning.ipynb

Objective:
- Load raw CSV data
- Perform deterministic, reproducible cleaning
- Fix schema, missing values, invalid entries
- Save cleaned (but not feature-engineered) data to data/interim/

Rules:
- No feature engineering
- No target-based transformations
- No distributional assumptions
"""


In [2]:
import os
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [4]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
data_path = os.path.join(project_root,'data', 'raw', 'crime_data_10000.csv')

crime_raw = pd.read_csv(data_path)
crime_raw.shape

(10000, 10)

In [5]:
crime_data = crime_raw.copy()

In [6]:
crime_data.columns = (
    crime_data.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "")
    .str.replace(")", "")
)

crime_data.columns

Index(['crime_id', 'crime_type', 'latitude', 'longitude', 'hour',
       'day_of_week', 'victim_age', 'suspect_age', 'weapon_used',
       'arrest_made'],
      dtype='object')

In [7]:
crime_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   crime_id     10000 non-null  int64  
 1   crime_type   10000 non-null  object 
 2   latitude     10000 non-null  float64
 3   longitude    10000 non-null  float64
 4   hour         10000 non-null  int64  
 5   day_of_week  10000 non-null  object 
 6   victim_age   10000 non-null  int64  
 7   suspect_age  10000 non-null  int64  
 8   weapon_used  7525 non-null   object 
 9   arrest_made  10000 non-null  int64  
dtypes: float64(2), int64(5), object(3)
memory usage: 781.4+ KB


In [8]:
numeric_candidates = ["age", "income", "crime_severity"]

for col in numeric_candidates:
    if col in crime_data.columns:
        crime_data[col] = pd.to_numeric(crime_data[col], errors="coerce")

In [10]:
missing_report = crime_data.isna().mean().sort_values(ascending=False)
missing_report

weapon_used    0.2475
crime_id       0.0000
crime_type     0.0000
latitude       0.0000
longitude      0.0000
hour           0.0000
day_of_week    0.0000
victim_age     0.0000
suspect_age    0.0000
arrest_made    0.0000
dtype: float64

In [11]:
categorical_cols = crime_data.select_dtypes(include=['object']).columns

crime_data[categorical_cols] = crime_data[categorical_cols].replace(['', "N/A", "NA", "null", "None"], np.nan)


In [13]:
before = crime_data.shape
crime_data = crime_data.drop_duplicates()
after = crime_data.shape[0]

before, after

((10000, 10), 10000)

In [14]:
if 'age' in crime_data.columns:
    crime_data.loc[(crime_data['age'] < 0) | (crime_data['age'] > 120), 'age'] = np.nan

if 'crime_severity' in crime_data.columns:
    crime_data.loc[crime_data['crime_severity'] < 0, 'crime_severity'] = np.nan

In [15]:
if "gender" in crime_data.columns:
    crime_data["gender"] = (
        crime_data["gender"]
        .str.lower()
        .str.strip()
        .replace({
            'm': "male",
            "f": "female"
        })
    )

In [16]:
id_cols = [col for col in crime_data.columns if col.endswith("_id")]

id_cols

['crime_id']

In [17]:
crime_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   crime_id     10000 non-null  int64  
 1   crime_type   10000 non-null  object 
 2   latitude     10000 non-null  float64
 3   longitude    10000 non-null  float64
 4   hour         10000 non-null  int64  
 5   day_of_week  10000 non-null  object 
 6   victim_age   10000 non-null  int64  
 7   suspect_age  10000 non-null  int64  
 8   weapon_used  7525 non-null   object 
 9   arrest_made  10000 non-null  int64  
dtypes: float64(2), int64(5), object(3)
memory usage: 781.4+ KB


In [18]:
crime_data["weapon_used"].isna().sum()

np.int64(2475)

In [22]:
pd.crosstab(
    crime_data["crime_type"],
    crime_data["weapon_used"].isna(),
    normalize = "index"
)

weapon_used,False,True
crime_type,,
Assault,0.753501,0.246499
Burglary,0.750521,0.249479
Fraud,0.762978,0.237022
Homicide,0.747376,0.252624
Robbery,0.739037,0.260963
Theft,0.761771,0.238229
Vandalism,0.751753,0.248247


In [23]:
crime_data["weapon_used"] = crime_data["weapon_used"].fillna("unknown")

In [24]:
crime_data["weapon_used"] = (
    crime_data["weapon_used"]
    .str.lower()
    .str.strip()
)

In [25]:
INTERIM_PATH = "../data/interim/cleaned_data.csv"
crime_data.to_csv(INTERIM_PATH, index=False)

INTERIM_PATH

'../data/interim/cleaned_data.csv'